# Cross-Industry Accelerator — Create Data Agents
### Auto-Create Fabric IQ QA and Operations Agents

This notebook creates **two agents** backed by the ontology and data sources:

| Agent | Purpose | Power |
|---|---|---|
| **QA Agent** | Answer ad-hoc data questions in natural language | Queries Lakehouse + Eventhouse via ontology |
| **Ops Agent** | Monitor operations and surface alerts/anomalies | Real-time event analysis via ontology |

**What it does:**
1. Reads industry configuration from `00_Industry_Config`
2. Resolves the ontology item ID from the workspace
3. Creates the QA Agent item linked to the ontology
4. Creates the Ops Agent item linked to the ontology
5. Configures each agent with industry-appropriate instructions

> **Prerequisites:**
> 1. Run `05_Create_Ontology.ipynb` to create the ontology first
> 2. Run `04_Create_Semantic_Model.ipynb` to create the semantic model
> 3. Data must be loaded (Lakehouse + Eventhouse) for agents to query

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RUN CONFIGURATION NOTEBOOK
# ════════════════════════════════════════════════════════════════════════

In [ ]:
%run 00_Industry_Config

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RESOLVE WORKSPACE ITEMS
# ════════════════════════════════════════════════════════════════════════

import sempy.fabric as fabric
import json, requests

workspace_id = fabric.get_workspace_id()
access_token = notebookutils.credentials.getToken('pbi')
items_df = fabric.list_items()

# Resolve Ontology item ID
ont_matches = items_df[(items_df["Type"] == "Ontology") & (items_df["Display Name"] == ONTOLOGY_NAME)]
if ont_matches.empty:
    print(f"⚠ Ontology '{ONTOLOGY_NAME}' not found. Run 05_Create_Ontology first.")
    print(f"Available ontologies:")
    print(items_df[items_df["Type"] == "Ontology"][["Display Name", "Id"]].to_string(index=False))
    ontology_item_id = None
else:
    ontology_item_id = str(ont_matches.iloc[0].Id)
    print(f"✓ Ontology: {ONTOLOGY_NAME} → {ontology_item_id}")

# Resolve Lakehouse item ID
lh_matches = items_df[(items_df["Type"] == "Lakehouse") & (items_df["Display Name"] == LAKEHOUSE_NAME)]
lakehouse_item_id = str(lh_matches.iloc[0].Id) if not lh_matches.empty else None
print(f"✓ Lakehouse: {LAKEHOUSE_NAME} → {lakehouse_item_id}")

# Resolve Eventhouse item ID
eventhouse_item_id = None
if EVENTHOUSE_NAME:
    eh_matches = items_df[(items_df["Type"] == "Eventhouse") & (items_df["Display Name"] == EVENTHOUSE_NAME)]
    if not eh_matches.empty:
        eventhouse_item_id = str(eh_matches.iloc[0].Id)
        print(f"✓ Eventhouse: {EVENTHOUSE_NAME} → {eventhouse_item_id}")

print(f"\n  QA Agent name:  {DATA_AGENT_NAME}")
print(f"  Ops Agent name: {OPS_AGENT_NAME}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# INSTALL FABRIC DATA AGENT SDK
# ════════════════════════════════════════════════════════════════════════

import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "fabric-data-agent-sdk", "-q"])

from fabric.dataagent.client import (
    FabricDataAgentManagement,
    create_data_agent,
    delete_data_agent,
)

print("fabric-data-agent-sdk installed and imported.")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CREATE QA AGENT + ADD DATA SOURCES (via fabric-data-agent-sdk)
# ════════════════════════════════════════════════════════════════════════

# Create QA Data Agent
print(f"Creating QA Agent: {DATA_AGENT_NAME}...")
try:
    qa_agent = create_data_agent(DATA_AGENT_NAME)
    print(f"  Created: {DATA_AGENT_NAME}")
except Exception as e:
    if "already" in str(e).lower():
        print(f"  Agent already exists. Connecting to existing...")
        qa_agent = FabricDataAgentManagement(DATA_AGENT_NAME)
    else:
        print(f"  Error: {e}")
        qa_agent = None

if qa_agent:
    # Update instructions
    qa_instructions = (
        f"You are a data analyst agent for the {INDUSTRY} industry. "
        f"Answer questions about {INDUSTRY} data using the connected data sources. "
        f"Query dimension tables for reference data and fact tables for metrics. "
        f"When answering, cite specific data points and provide context. "
        f"You are scoped to {INDUSTRY} data only."
    )
    qa_agent.update_configuration(instructions=qa_instructions)
    print(f"  Instructions set.")

    # Add Lakehouse as data source
    print(f"  Adding Lakehouse: {LAKEHOUSE_NAME}...")
    try:
        qa_agent.add_datasource(LAKEHOUSE_NAME, type="lakehouse")
        print(f"  Lakehouse added.")
    except Exception as e:
        print(f"  Lakehouse: {e}")

    # Add Warehouse as data source
    print(f"  Adding Warehouse: {WAREHOUSE_NAME}...")
    try:
        qa_agent.add_datasource(WAREHOUSE_NAME, type="warehouse")
        print(f"  Warehouse added.")
    except Exception as e:
        print(f"  Warehouse: {e}")

    # Add Eventhouse/KQL Database as data source
    if EVENTHOUSE_DATABASE:
        print(f"  Adding KQL Database: {EVENTHOUSE_DATABASE}...")
        try:
            qa_agent.add_datasource(EVENTHOUSE_DATABASE, type="kqldatabase")
            print(f"  KQL Database added.")
        except Exception as e:
            print(f"  KQL DB: {e}")

    # Publish the agent
    print(f"  Publishing...")
    try:
        qa_agent.publish()
        print(f"  QA Agent published: {DATA_AGENT_NAME}")
    except Exception as e:
        print(f"  Publish: {e}")

    # Show config
    try:
        config = qa_agent.get_configuration()
        print(f"  Config: {config}")
        datasources = qa_agent.get_datasources()
        print(f"  Data sources: {len(datasources)}")
    except Exception as e:
        print(f"  Get config: {e}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CREATE OPS AGENT + ADD DATA SOURCES (via fabric-data-agent-sdk)
# ════════════════════════════════════════════════════════════════════════

print(f"\nCreating Ops Agent: {OPS_AGENT_NAME}...")
try:
    ops_agent = create_data_agent(OPS_AGENT_NAME)
    print(f"  Created: {OPS_AGENT_NAME}")
except Exception as e:
    if "already" in str(e).lower():
        print(f"  Agent already exists. Connecting to existing...")
        ops_agent = FabricDataAgentManagement(OPS_AGENT_NAME)
    else:
        print(f"  Error: {e}")
        ops_agent = None

if ops_agent:
    # Update instructions
    ops_instructions = (
        f"You are an operations monitoring agent for the {INDUSTRY} industry. "
        f"Analyze event streams and fact tables to detect anomalies and surface alerts. "
        f"Focus on KPIs, SLA compliance, quality metrics, and risk indicators. "
        f"When reporting issues, provide severity and recommended actions. "
        f"You are scoped to {INDUSTRY} operational data only."
    )
    ops_agent.update_configuration(instructions=ops_instructions)
    print(f"  Instructions set.")

    # Add Warehouse as data source
    print(f"  Adding Warehouse: {WAREHOUSE_NAME}...")
    try:
        ops_agent.add_datasource(WAREHOUSE_NAME, type="warehouse")
        print(f"  Warehouse added.")
    except Exception as e:
        print(f"  Warehouse: {e}")

    # Add Eventhouse/KQL Database as data source
    if EVENTHOUSE_DATABASE:
        print(f"  Adding KQL Database: {EVENTHOUSE_DATABASE}...")
        try:
            ops_agent.add_datasource(EVENTHOUSE_DATABASE, type="kqldatabase")
            print(f"  KQL Database added.")
        except Exception as e:
            print(f"  KQL DB: {e}")

    # Publish the agent
    print(f"  Publishing...")
    try:
        ops_agent.publish()
        print(f"  Ops Agent published: {OPS_AGENT_NAME}")
    except Exception as e:
        print(f"  Publish: {e}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# AGENT CREATION SUMMARY
# ════════════════════════════════════════════════════════════════════════

print(f"\n{'═'*60}")
print(f"DATA AGENT SUMMARY — {INDUSTRY.upper()}")
print(f"{'═'*60}")
print(f"")
print(f"  Ontology:       {ONTOLOGY_NAME}")
print(f"  Lakehouse:      {LAKEHOUSE_NAME}")
print(f"  Eventhouse:     {EVENTHOUSE_NAME or 'N/A'}")
print(f"")
print(f"  QA Agent:       {DATA_AGENT_NAME}")
print(f"    → Answers ad-hoc data questions about {INDUSTRY} data")
print(f"    → Queries: dimensions, facts, aggregates")
print(f"")
print(f"  Ops Agent:      {OPS_AGENT_NAME}")
print(f"    → Monitors real-time events and operational metrics")
print(f"    → Surfaces: anomalies, SLA breaches, quality alerts")
print(f"")
print(f"✅ Agent creation complete.")
print(f"   Next: Run 07_Create_Dashboards.ipynb to create real-time and Power BI dashboards.")